# Assignment 5


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

input_img = cv2.imread('Sunflower_from_Silesia2.jpg', cv2.IMREAD_GRAYSCALE)
low_contrast_img = np.uint8(np.clip(input_img * 0.5 + 50, 0, 255))


## Question 1: Histogram Equalization


In [ ]:
def manual_hist_equalization(image):
    h, w = image.shape
    total_pixels = h * w
    hist = np.zeros(256, dtype=int)
    for pixel in image.ravel():
        hist[pixel] += 1
    pdf = hist / total_pixels
    cdf = np.cumsum(pdf)
    transform_map = np.round(255 * cdf).astype(np.uint8)
    equalized_img = transform_map[image]
    return equalized_img, hist, transform_map

eq_manual, orig_hist, t_map = manual_hist_equalization(low_contrast_img)
eq_cv2 = cv2.equalizeHist(low_contrast_img)
eq_hist, _ = np.histogram(eq_manual.flatten(), bins=256, range=[0, 256])

plt.figure(figsize=(14, 8))

plt.subplot(2, 2, 1)
plt.title('Original (Low Contrast) Image')
plt.imshow(low_contrast_img, cmap='gray')
plt.axis('off')

plt.subplot(2, 2, 2)
plt.title('Equalized Image')
plt.imshow(eq_manual, cmap='gray')
plt.axis('off')

plt.subplot(2, 2, 3)
plt.title('Original Histogram')
plt.bar(range(256), orig_hist, color='steelblue', width=1)
plt.xlim([0, 255])
plt.xlabel('Intensity')
plt.ylabel('Pixel Count')

plt.subplot(2, 2, 4)
plt.title('Equalized Histogram')
plt.bar(range(256), eq_hist, color='coral', width=1)
plt.xlim([0, 255])
plt.xlabel('Intensity')
plt.ylabel('Pixel Count')

plt.tight_layout()
plt.show()


## Question 2: Histogram Matching


In [ ]:
def histogram_matching(source, reference):
    h_s, w_s = source.shape
    h_r, w_r = reference.shape

    hist_s, _ = np.histogram(source.flatten(), 256, [0, 256])
    cdf_s = np.cumsum(hist_s) / (h_s * w_s)

    hist_r, _ = np.histogram(reference.flatten(), 256, [0, 256])
    cdf_r = np.cumsum(hist_r) / (h_r * w_r)

    mapping = np.zeros(256, dtype=np.uint8)
    for src_level in range(256):
        diff = np.abs(cdf_r - cdf_s[src_level])
        mapping[src_level] = np.argmin(diff)

    matched_img = mapping[source]
    return matched_img, hist_s, hist_r

reference_img = np.uint8(np.clip(np.power(input_img / 255.0, 0.4) * 255.0, 0, 255))
matched_img, hist_src, hist_ref = histogram_matching(low_contrast_img, reference_img)
hist_matched, _ = np.histogram(matched_img.flatten(), 256, [0, 256])

plt.figure(figsize=(15, 8))

plt.subplot(2, 3, 1)
plt.title('Input Source Image')
plt.imshow(low_contrast_img, cmap='gray')
plt.axis('off')

plt.subplot(2, 3, 2)
plt.title('Target Reference Image')
plt.imshow(reference_img, cmap='gray')
plt.axis('off')

plt.subplot(2, 3, 3)
plt.title('Histogram Matched Image')
plt.imshow(matched_img, cmap='gray')
plt.axis('off')

plt.subplot(2, 3, 4)
plt.title('Source Histogram')
plt.bar(range(256), hist_src, color='steelblue', width=1)
plt.xlim([0, 255])

plt.subplot(2, 3, 5)
plt.title('Reference Histogram')
plt.bar(range(256), hist_ref, color='darkgreen', width=1)
plt.xlim([0, 255])

plt.subplot(2, 3, 6)
plt.title('Matched Histogram')
plt.bar(range(256), hist_matched, color='crimson', width=1)
plt.xlim([0, 255])

plt.tight_layout()
plt.show()
